In [35]:
"""
Score a list of texts against a query subset using jina-embeddings-v5-text-small.

Edit `texts` and `query_indices` below, then run:
    python score_collection.py
"""

import numpy as np
from sentence_transformers import SentenceTransformer, util
from sklearn.preprocessing import normalize
import torch


def USE_sim(p, vs):
    return 1 - torch.arccos(
                            torch.clamp(
                                util.cos_sim(p, vs), -1, 1
                            )
                        )/torch.pi
# ---------------------------------------------------------------------------
# Your data — edit these
# ---------------------------------------------------------------------------

texts = [
    "Hond.",
    "De Armeense genocide met veel slachtoffers.",
    "Theepot van zilver, vervaardigd in Amsterdam rond 1760.",
    "Bord van Delfts aardewerk met blauwe decoratie.",
    "Zilveren kandelaar met gegraveerde bloemmotieven.",
    "Houten stoel met gebogen poten, 19e eeuw.",
    "Porseleinen vaas met Japanse invloeden, Meissen.",
]

# Indices into `texts` that form the query set
query_indices = [0, 6]

# "mean": best for a coherent query set; "max": best for a diverse one
aggregation = "max"

top_k = 10

# ---------------------------------------------------------------------------
# Embed
# ---------------------------------------------------------------------------

# Task-specific variant: LoRA adapter pre-merged, no trust_remote_code needed
model = SentenceTransformer("jinaai/jina-embeddings-v5-text-small-text-matching")

embeddings = model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
)
embeddings = normalize(embeddings)  # unit-normalise → cosine == dot product

# ---------------------------------------------------------------------------
# Score
# ---------------------------------------------------------------------------

query_embeddings = embeddings[query_indices]      # (Q, D)
sim_matrix = embeddings @ query_embeddings.T      # (N, Q)
sim_matrix = USE_sim(query_embeddings, embeddings).T


if aggregation == "mean":
    scores = sim_matrix.mean(axis=1)
elif aggregation == "max":
    scores = sim_matrix.max(axis=1).values

# Exclude the query objects themselves
ranked = [i for i in np.argsort(-scores) if i not in set(query_indices)]

# ---------------------------------------------------------------------------
# Print results
# ---------------------------------------------------------------------------

print(f"\nQuery texts:")
for i in query_indices:
    print(f"  [{i}] {texts[i]}")

print(f"\nTop-{top_k} results ({aggregation} aggregation):")
for rank, idx in enumerate(ranked[:top_k], 1):
    print(f"  {rank}. [{idx}] score={scores[idx]:.4f}  {texts[idx]}")

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Query texts:
  [0] Hond.
  [6] Porseleinen vaas met Japanse invloeden, Meissen.

Top-10 results (max aggregation):
  1. [0] score=1.0000  Hond.
  2. [6] score=1.0000  Porseleinen vaas met Japanse invloeden, Meissen.
  3. [3] score=0.7085  Bord van Delfts aardewerk met blauwe decoratie.
  4. [2] score=0.6908  Theepot van zilver, vervaardigd in Amsterdam rond 1760.
  5. [4] score=0.6858  Zilveren kandelaar met gegraveerde bloemmotieven.
  6. [5] score=0.6830  Houten stoel met gebogen poten, 19e eeuw.
  7. [1] score=0.6362  De Armeense genocide met veel slachtoffers.


In [ ]:

Top-10 results (max aggregation):
  1. [6] score=1.0000  Porseleinen vaas met Japanse invloeden, Meissen.
  2. [3] score=0.6786  Bord van Delfts aardewerk met blauwe decoratie.
  3. [4] score=0.6303  Zilveren kandelaar met gegraveerde bloemmotieven.
  4. [2] score=0.5395  Theepot van zilver, vervaardigd in Amsterdam rond 1760.
  5. [5] score=0.3852  Houten stoel met gebogen poten, 19e eeuw.
  6. [0] score=0.3168  Hond.
  7. [1] score=0.1601  De Armeense genocide met veel slachtoffers.

In [27]:
scores

array([1.0000002 , 0.41483372, 0.5642717 , 0.6092292 , 0.5509938 ,
       0.5438607 , 1.0000004 ], dtype=float32)

In [32]:
USE_sim(query_embeddings, embeddings).T.max(axis=1).values


tensor([1.0000, 0.6362, 0.6908, 0.7085, 0.6858, 0.6830, 1.0000])

In [21]:
sim_matrix

array([[1.0000002 , 0.5138339 ],
       [0.41483372, 0.40152735],
       [0.43212682, 0.5642717 ],
       [0.4566411 , 0.6092292 ],
       [0.46138048, 0.5509938 ],
       [0.48757523, 0.5438607 ],
       [0.5138338 , 1.0000004 ]], dtype=float32)